In [1]:
!pip install -q torch transformers sentence-transformers faiss-cpu pandas tqdm scikit-learn rouge-score nltk bert-score sacrebleu
# 7_prompt_engineering_comparison.py

# ------------------- IMPORTS -------------------
import pandas as pd, numpy as np, torch, faiss, time, nltk, warnings, logging
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score
from sacrebleu.metrics import BLEU

# Reduce HuggingFace model init warnings
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# NLTK downloads
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

warnings.filterwarnings('ignore')

# ------------------- DATA -------------------
df = pd.read_csv('/kaggle/input/mlops-amazon/amazon.csv')

documents = [
    f"""Product: {r['product_name']}
Price: {r['discounted_price']} | Rating: {r['rating']} ({r['rating_count']} reviews)
Description: {r['about_product']}""" 
    for _, r in df.iterrows()
]

TEST_QUERIES = [
{
"query": "Recommend a good fast charging USB-C cable under 300 rupees",
"reference": "The boAt A400 at ₹299 is recommended, supporting 24W-25W fast charging and having a slightly higher customer rating. For a tighter budget, the pTron Solero TB301 at ₹149 supports 15W fast charging."
},
{
"query": "Which cable has the highest rating and supports 60W charging?",
"reference": "The Belkin USB-C to USB-C Fast Charging Cable (60W PD) is the best match, offering a strong 4.5-star rating along with full 60W fast-charging support."
},
{
"query": "What is the best iPhone lightning cable in the list?",
"reference": "For super-fast charging with a USB-C adapter, the Belkin Lightning to USB-C is best. For a reliable and durable standard cable with a good warranty, the Duracell USB-A to Lightning is a great choice. The Hi-Mobiler is the most budget-friendly option."
},
{
"query": "Suggest me some good long lasting headphones",
"reference": "The boAt Bassheads 100 in-ear wired earphones are recommended for longevity, featuring a premium coated wire for sturdiness and a 1-year warranty. They have a 4.1-star rating from over 360,000 reviews and cost between ₹349-₹379."
}
]

# ------------------- METRICS CLASS -------------------
class Metrics:
    def __init__(self):
        self.rouge = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)
        self.bleu = BLEU(effective_order=True)
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

    def all(self, pred, ref, ctx):
        r = self.rouge.score(ref, pred)
        metrics = {
            'rouge_1_f1': r['rouge1'].fmeasure,
            'rouge_l_f1': r['rougeL'].fmeasure,
            'bleu': self.bleu.sentence_score(pred, [ref]).score / 100,
            'meteor': meteor_score([word_tokenize(ref.lower())], word_tokenize(pred.lower())),
        }
        P, R, F = bert_score([pred], [ref], model_type="microsoft/deberta-large-mnli", verbose=False)
        metrics['bert_f1'] = F.mean().item()

        e1 = self.embedder.encode(pred)
        e2 = self.embedder.encode(ref)
        metrics['emb_sim'] = util.cos_sim(e1, e2).item()

        c_emb = self.embedder.encode(ctx)
        metrics['faith'] = min(1.0, 0.7 * util.cos_sim(e1, c_emb).item() + 0.3)

        return metrics

    def composite(self, m):
        w = {
            'rouge_1_f1': 0.1,
            'rouge_l_f1': 0.1,
            'bleu': 0.1,
            'meteor': 0.15,
            'bert_f1': 0.25,
            'emb_sim': 0.2,
            'faith': 0.1
        }
        return sum(m[k] * w[k] for k in w)

metrics_calc = Metrics()

# ------------------- RAG CLASS -------------------
class RAG:
    def __init__(self, emb_name, generator):
        self.emb_name = emb_name
        self.generator = generator

        print(f"Loading embedding model: {emb_name}")
        self.embedder = SentenceTransformer(emb_name)

        dim = self.embedder.encode(["test"]).shape[1]
        self.index = faiss.IndexFlatIP(dim)

        print(f"Embedding {len(documents)} documents...")
        batches = [documents[i:i+32] for i in range(0, len(documents), 32)]
        for b in tqdm(batches, desc="Indexing"):
            embs = self.embedder.encode(b, normalize_embeddings=True)
            self.index.add(embs)

    def retrieve(self, q, k):
        qe = self.embedder.encode([q], normalize_embeddings=True)
        D, I = self.index.search(qe, k)
        ctx = "\n\n".join([documents[i] for i in I[0]])
        return ctx

# ------------------- PATCH GENERATE METHOD -------------------
def generate(self, q, ctx, prompt_template):
    prompt = prompt_template.format(ctx=ctx, q=q)
    out = self.generator(
        prompt,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.95,
        top_k=50,
        do_sample=True
    )[0]['generated_text']
    ans = out.split("Answer:")[-1].strip() if "Answer:" in out else out.strip()
    return ans

RAG.generate = generate

# ------------------- LOAD GENERATOR AND RAG -------------------
GEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

print("\nLoading generator...")
generator = pipeline(
    "text-generation",
    model=GEN_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("\nLoading RAG with fixed models...")
rag = RAG(EMBEDDING_MODEL, generator)

# ------------------- PROMPT ENGINEERING EXPERIMENT -------------------
results = []
PROMPT_TEMPLATES = [
    "Context:\n{ctx}\n\nQuestion: {q}\nAnswer:",
    "Answer briefly using only the context:\n{ctx}\nQ: {q}\nA:",
    "You are a helpful shopping assistant. Use only the given products:\n{ctx}\nQuestion: {q}\nAnswer:",
    "Based on the Amazon products below, recommend the best one:\n{ctx}\nQuery: {q}\nBest choice:"
]

for idx, template in enumerate(PROMPT_TEMPLATES, 1):
    print(f"\n{'='*80}\nTESTING PROMPT TEMPLATE {idx}: {template[:50]}...\n{'='*80}")
    for qd in TEST_QUERIES:
        ctx = rag.retrieve(qd["query"], k=5)
        ans = rag.generate(qd["query"], ctx, prompt_template=template)
        m = metrics_calc.all(ans, qd["reference"], ctx)
        m['composite'] = metrics_calc.composite(m)
        results.append({**m, "prompt_id": idx, "query": qd["query"][:60]})
        print("\n------------------------------------------------------------")
        print(f"Prompt ID: {idx}")
        print(f"Query: {qd['query']}")
        print("\nGenerated Answer:")
        print(ans)
        print(f"\nComposite Score: {m['composite']:.4f}")
        print("------------------------------------------------------------\n")

df_out = pd.DataFrame(results)
summary = df_out.groupby("prompt_id")["composite"].mean().sort_values(ascending=False)

print("\n================ FINAL SUMMARY ================\n")
print("Average Composite Scores by Prompt Template:")
print(summary)

best_prompt = summary.idxmax()
best_score = summary.max()
print(f"\n🏆 Best Prompt ID: {best_prompt} → Composite Score: {best_score:.4f}")

df_out.to_csv("7_prompt_engineering_comparison.csv", index=False)
print("\nPrompt engineering comparison results saved → 7_prompt_engineering_comparison.csv")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

2025-12-05 08:04:17.015306: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764921857.198231      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764921857.251574      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...


------------------------------------------------------------
Prompt ID: 4
Query: Which cable has the highest rating and supports 60W charging?

Generated Answer:
Based on the Amazon products below, recommend the best one:
Product: MI Xiaomi USB Type C HYperCharge Cable 6A 100cm Sturdy and Durable Black Supports 120W HyperCharging
Price: ₹499 | Rating: 4.3 (30,411 reviews)
Description: Supports 120W Fast Charging|High Quality Design

Product: Ambrane 60W / 3A Type C Fast Charging Unbreakable 1.5m L Shaped Braided Cable, PD Technology, 480Mbps Data Transfer for Smartphones, Tablet, Laptops & other type c devices (ABLC10, Black)
Price: ₹179 | Rating: 4.0 (1,934 reviews)
Description: Stay ahead and never miss out with a 3A and 30W fast charging for your devices.|It supports Quick Charge 2.0/ 3.0 technology to keep your devices boosted up.|Its L-shape provides durability and comfort while you charge your favourite gadgets.|It comes with upto 60W flash charge support to make you always stay

The following layers were not sharded: encoder.layer.*.attention.output.LayerNorm.weight, embeddings.LayerNorm.weight, encoder.layer.*.attention.self.pos_proj.weight, embeddings.word_embeddings.weight, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.self.pos_q_proj.weight, encoder.layer.*.attention.self.q_bias, embeddings.LayerNorm.bias, encoder.layer.*.attention.self.in_proj.weight, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.self.v_bias, encoder.rel_embeddings.weight, encoder.layer.*.attention.self.pos_q_proj.bias, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.output.dense.weight, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.output.dense.bias, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.attention.output.LayerNorm.bias



------------------------------------------------------------
Prompt ID: 4
Query: What is the best iPhone lightning cable in the list?

Generated Answer:
Based on the Amazon products below, recommend the best one:
Product: Hi-Mobiler iPhone Charger Lightning Cable,2 Pack Apple MFi Certified USB iPhone Fast Chargering Cord,Data Sync Transfer for 13/12/11 Pro Max Xs X XR 8 7 6 5 5s iPad iPod More Model Cell Phone Cables
Price: ₹254 | Rating: 4.0 (2,905 reviews)
Description: Internationally Certified Materials And Exquisite Design Safe Fast Charging Cables: This iPhone charger cable are made of high purity four-core copper core and smart intelligent chip and high-quality TPE ,with overcharge protection, stable current protection, automatic switching and battery protection design.|MFI Certified Lightning Cable: Each cable contains an authorization chip issued to ensure 100% compatibility with iPhone.Built with terminal and smart chip. connector with a lightning end, ensured safe charging f

The following layers were not sharded: encoder.layer.*.attention.output.LayerNorm.weight, embeddings.LayerNorm.weight, encoder.layer.*.attention.self.pos_proj.weight, embeddings.word_embeddings.weight, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.self.pos_q_proj.weight, encoder.layer.*.attention.self.q_bias, embeddings.LayerNorm.bias, encoder.layer.*.attention.self.in_proj.weight, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.self.v_bias, encoder.rel_embeddings.weight, encoder.layer.*.attention.self.pos_q_proj.bias, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.output.dense.weight, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.output.dense.bias, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.attention.output.LayerNorm.bias



------------------------------------------------------------
Prompt ID: 4
Query: Suggest me some good long lasting headphones

Generated Answer:
Based on the Amazon products below, recommend the best one:
Product: boAt Bassheads 152 in Ear Wired Earphones with Mic(Active Black)
Price: ₹449 | Rating: 4.1 (91,770 reviews)
Description: Break away from old habits through HD sound via 10mm drivers, crystal clear sound to your ears helps you execute what you have visualized perfectly, enhance your senses with the BassHeads 152.|Vibe your rhythm all day with fantastic bass heavy tunes that drown out your stress and brings back your search for the ultimate quest, it’s time to get kicking.|Communicate seamlessly with the inbuilt mic that has high sensitivity and a versatile frequency range to provide the ultimate experience; Foldable/Collapsible: Yes, Deep Bass: Yes With Microphone: Yes.|Durable and highly resistant, the braided cable adds to the nirvana with its soft feel and tangle resilient